In [ ]:
# ERDOS L40S — HOT CORRIDOR SIEVE (Kaggle T4)
import json, time, math, os, subprocess
from datetime import datetime
from pathlib import Path

try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    GPU = r.stdout.strip() if r.returncode==0 else 'CPU'
except: GPU = 'CPU'

print(f"GPU: {GPU}")
print(f"Start: {datetime.now().isoformat()}")

In [ ]:
# CONFIG
OUTPUT = Path("/kaggle/working/erdos_output.json")
CHUNK_SIZE = 100_000_000   # 100M per run (Kaggle T4, 9hr limit)
HOT_MOD9 = {0, 3, 6}
SAVE_INTERVAL = 250_000     # Save every 250K candidates
RUN_START = time.time()

print(f"Target: mod24=0, mod9 in {HOT_MOD9}")
print(f"Chunk: {CHUNK_SIZE:,} per run")

In [ ]:
# INTEGER SOLVER — parametric identities + divisor search
def erdos_straus_int(n):
    triples = set()

    # Identity 1: n=4k -> x=y=z=3k
    if n % 4 == 0:
        k = n // 4
        triples.add((3*k, 3*k, 3*k))

    # Identity 2: n=3k -> (2k, 2k, n)
    if n % 3 == 0:
        k = n // 3
        triples.add((2*k, 2*k, n))

    # Divisor search for additional solutions
    sqrt_n = int(n**0.5)
    for d in range(1, min(sqrt_n, 300)):
        if n % d != 0:
            continue
        for divisor in (d, n // d):
            if divisor > 10_000_000:
                continue
            x = divisor
            num = 4*x - n
            if num <= 0:
                continue
            den = n * x
            g = math.gcd(num, den)
            a = num // g
            b = den // g

            if a == 2:
                y = z = b
                if y >= x and z >= y:
                    triples.add((x, y, z))
            elif a == 1:
                y = b + 1
                z = b * (b + 1)
                if y >= x and z >= y:
                    triples.add((x, y, z))
            elif a == 3 and b % 2 == 0:
                y = b
                z = b // 2
                if y >= x and z >= y:
                    triples.add((x, y, z))

    return sorted(triples, key=lambda t: (t[0], t[1], t[2]))

def erdos_straus(n):
    mod9 = n % 9
    if n % 24 != 0:
        return False, "SKIP", (), 0
    if mod9 not in HOT_MOD9:
        return False, "SKIP", (), 0
    triples = erdos_straus_int(n)
    if triples:
        depth = "BREACH_MOD9" if mod9 in (0, 3, 6) else "STABLE_MOD9"
        return True, depth, triples[0], len(triples)
    else:
        return False, "ANOMALY", (), 0

print('Solver ready')

In [ ]:
# LOAD PRIOR STATE (if resuming)
if OUTPUT.exists():
    state = json.loads(OUTPUT.read_text())
    start_n = state.get("last_n", 32_000_000)
    solutions = state.get("solutions", [])
    stats = state.get("stats", {"stable": 0, "breach": 0, "neutral": 0, "total_checked": 0})
    print(f"Resuming from n={start_n:,} — {len(solutions)} existing solutions")
else:
    start_n = 32_000_000
    solutions = []
    stats = {"stable": 0, "breach": 0, "neutral": 0, "total_checked": 0}
    print(f"Fresh start from n={start_n:,}")

chunk_candidates_base = stats["total_checked"]
candidates_at_last_save = stats["total_checked"]

In [ ]:
# MAIN LOOP — stride by 24
n0 = ((start_n + 23) // 24) * 24
if n0 < start_n:
    n0 += 24
end_n = start_n + CHUNK_SIZE
checkpoint_time = time.time()
anomalies = []

print(f"Stride-24: {n0:,} -> {end_n:,} ({(end_n - n0) // 24:,} candidates)")

try:
    for n in range(n0, end_n, 24):
        has_sol, depth, triple, num_sol = erdos_straus(n)
        stats["total_checked"] += 1

        if has_sol:
            entry = {
                "n": n, "mod9": n % 9, "mod24": n % 24,
                "depth": depth, "triple": list(triple),
                "num_solutions": num_sol,
                "timestamp": datetime.now().isoformat()
            }
            solutions.append(entry)
            if "STABLE" in depth: stats["stable"] += 1
            elif "BREACH" in depth: stats["breach"] += 1
            else: stats["neutral"] += 1
        elif depth == "ANOMALY":
            anomalies.append(n)
            stats["neutral"] += 1

        # Periodic save
        candidates_since_save = stats["total_checked"] - candidates_at_last_save
        if candidates_since_save >= SAVE_INTERVAL:
            interval_elapsed = time.time() - checkpoint_time
            rate = candidates_since_save / interval_elapsed if interval_elapsed > 0 else 0

            state = {
                "last_n": n, "solutions": solutions[-500:],
                "stats": stats, "rate_per_sec": round(rate),
                "timestamp": datetime.now().isoformat(),
                "gpu": GPU, "anomalies": len(anomalies)
            }
            OUTPUT.write_text(json.dumps(state))

            total_candidates = (end_n - n0) // 24
            chunk_checked = stats["total_checked"] - chunk_candidates_base
            progress_pct = chunk_checked / max(1, total_candidates) * 100
            print(f"  [{progress_pct:.1f}%] n={n:,} | {len(solutions)} sols | "
                  f"S:{stats['stable']} B:{stats['breach']} | {rate:.0f} cand/s")

            candidates_at_last_save = stats["total_checked"]
            checkpoint_time = time.time()

except KeyboardInterrupt:
    print("\nInterrupted — saving...")
except Exception as e:
    print(f"\nError: {e}")
    import traceback; traceback.print_exc()

In [ ]:
# FINAL SAVE + REPORT
final_state = {
    "last_n": end_n, "solutions": solutions, "stats": stats,
    "anomalies": anomalies, "completed_chunk": True,
    "timestamp": datetime.now().isoformat(),
    "gpu": GPU, "chunk_size": CHUNK_SIZE,
    "candidates_checked": stats["total_checked"]
}
OUTPUT.write_text(json.dumps(final_state))

total_runtime = time.time() - RUN_START
ct = stats["total_checked"]
hit_rate = len(solutions) / max(1, ct) * 100

print("\n" + "=" * 50)
print(f"ERDOS HOT CORRIDOR — DONE")
print(f"Runtime: {total_runtime/3600:.2f}h")
print(f"Candidates: {ct:,} | Range: ~{ct*24:,} raw n")
print(f"Solutions: {len(solutions)} ({hit_rate:.1f}%)")
print(f"STABLE: {stats['stable']} | BREACH: {stats['breach']}")
print(f"ANOMALIES: {len(anomalies)}")
if anomalies:
    print(f"  Values: {anomalies[:20]}")
print(f"Last n: {end_n:,}")
print("=" * 50)

# Inline JSON for cron balancer
print(json.dumps({
    "stable": stats["stable"], "breach": stats["breach"],
    "anomalies": len(anomalies), "total_solutions": len(solutions),
    "candidates_checked": ct, "hit_rate_pct": round(hit_rate, 2),
    "last_n": end_n, "runtime_h": round(total_runtime/3600, 2)
}))